# nb_22_silver_identity

Silver, part 3. Finds the **same person across regions** and gives every client record a `person_key`.

Why this matters: a client who shops in New York and in Paris has two client records, one per region. Without matching, the house sees two small clients instead of one valuable one.

How it works
1. **Prepare names-** Each client's *latest* version is used. Accents are removed, case is ignored, nicknames and initials are understood. Emails and phones from *every* version count as evidence.
2. **Find candidate pairs-** Two records from *different regions* become a candidate when they share an email, share a phone number, or have a compatible name.
3. **Score each pair with a rule-** Every pair gets the name of the rule that matched and a confidence between 0 and 1. Both are stored.
4. **Link only confident pairs-** Pairs at or above `AUTO_LINK_MIN` are linked. Weaker pairs are kept as `review` and are *not* merged. A pair that matches **only on the name** is ranked by how rare the name is: two people called "Yuto Ito" are probably different people, but two people with a very unusual name probably are not.
5. **Group linked records into people-** A person can never have two records in the same region. If a group breaks that rule, it is split and reported.
6. **Resolve consent-** If the regions disagree about cross-region sharing, the safest answer wins: sharing is allowed only if **every** record says yes.

Input: `silver_crm_client` (from nb_20; it keeps every version of a client)

Output: `silver_identity_pairs`, `silver_client_identity` (one row per client record), `silver_person` (one row per person), plus rows in `ctl_dq_results` and `ctl_run_log`

The answer key is not used here. Accuracy is measured separately in `nb_29_identity_eval`.


In [ ]:
# ---- Settings -------------------------------------------------------------------------
# Pairs with a confidence at or above this value are linked. Everything else stays "review".
AUTO_LINK_MIN = 0.80

# Rule name -> confidence. Evidence is checked from the strongest rule to the weakest.
RULE_CONFIDENCE = {
    "email_and_phone": 0.99,   # same email and same phone
    "email_and_name":  0.97,   # same email and a compatible name
    "phone_and_name":  0.92,   # same phone and a compatible name
    "email_only":      0.70,   # same email but the names do not fit (could be a family address)
    "phone_only":      0.60,   # same phone but the names do not fit (could be a shared number)
    "name_only_rare_name":   0.55,   # compatible name, nothing else, and the name is rare (review first)
    "name_only_common_name": 0.40,   # compatible name, nothing else, and the name is common (likely a different person)
}
# A name is "rare" when at most this many client records carry it (sorted first+last, accents removed).
# With the default of 2, a pair of records that are the only holders of that name counts as rare.
RARE_NAME_MAX = 2
# An email or phone used by more than 2 clients, or twice inside one region, is treated as shared.
# A shared identifier lowers a pair to this confidence, so it falls to "review".
SHARED_IDENTIFIER_CONFIDENCE = 0.55

# Nickname list. In production this would be a maintained reference table.
NICKNAMES = {"alexandra": "alex", "alexander": "alex", "katherine": "kate", "catherine": "kate",
             "elizabeth": "liz", "jennifer": "jenny", "michael": "mike", "christopher": "chris",
             "william": "bill", "robert": "rob", "stephanie": "steph", "nicholas": "nick",
             "margaret": "peggy", "richard": "rick", "jonathan": "jon", "samantha": "sam",
             "daniel": "dan", "matthew": "matt", "benjamin": "ben", "joseph": "joe"}

In [ ]:
# ---- Setup ----------------------------------------------------------------------------
import uuid
import unicodedata
from datetime import datetime, timezone
from functools import reduce

import pyspark.sql.functions as F
from pyspark.sql import Row, Window
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, LongType,
                               DoubleType, TimestampType, BooleanType)

SILVER_BATCH_ID = "S" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:6]
print("silver batch id:", SILVER_BATCH_ID)


def now_utc():
    return datetime.now(timezone.utc).replace(tzinfo=None)


dq_rows, run_rows, gaps = [], [], []


def add_dq(table, rule, severity, checked, failed):
    dq_rows.append(Row(batch_id=SILVER_BATCH_ID, layer="silver", table_name=table, rule=rule, severity=severity,
                       rows_checked=int(checked), rows_failed=int(failed),
                       pct_failed=round(100.0 * failed / checked, 3) if checked else 0.0, checked_at=now_utc()))


def write_table(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    return spark.table(name).count()


def log_run(obj, status, rows, started, message, failure_class=None):
    run_rows.append(Row(batch_id=SILVER_BATCH_ID, layer="silver", object_name=obj, status=status, files_loaded=0,
                        rows_loaded=int(rows), started_at=started, finished_at=now_utc(),
                        failure_class=failure_class, message=message))

In [ ]:
# ---- Name comparison ------------------------------------------------------------------
# These are plain Python functions so they are easy to test. They are wrapped as Spark UDFs below.
def fold(s):
    """Lower case, remove accents and everything that is not a letter."""
    if s is None:
        return None
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower()
    return "".join(ch for ch in s if ch.isalpha())


def edit_distance_le1(a, b):
    """True when a and b differ by at most one edit (insert, delete, replace) or one swap of neighbours."""
    if a == b:
        return True
    if abs(len(a) - len(b)) > 1:
        return False
    if len(a) == len(b):
        diffs = [i for i in range(len(a)) if a[i] != b[i]]
        if len(diffs) == 1:
            return True
        return len(diffs) == 2 and diffs[1] == diffs[0] + 1 and a[diffs[0]] == b[diffs[1]] and a[diffs[1]] == b[diffs[0]]
    short, long_ = (a, b) if len(a) < len(b) else (b, a)
    return any(long_[:i] + long_[i + 1:] == short for i in range(len(long_)))


def token_match(a, b):
    """How do two folded name parts relate? exact | typo | nickname | initial | None."""
    if not a or not b:
        return None
    if a == b:
        return "exact"
    if min(len(a), len(b)) >= 4 and edit_distance_le1(a, b):
        return "typo"
    if NICKNAMES.get(a) == b or NICKNAMES.get(b) == a:
        return "nickname"
    if min(len(a), len(b)) >= 3 and (a.startswith(b) or b.startswith(a)):
        return "nickname"
    if (len(a) == 1 and b.startswith(a)) or (len(b) == 1 and a.startswith(b)):
        return "initial"
    return None


def name_match_kind(f1, l1, f2, l2):
    """Compare two (first, last) names. Returns exact | typo | nickname | initial | swapped | None.
    Rule: at most one of the two parts may be fuzzy. Two fuzzy parts together are too weak to trust."""
    f1, l1, f2, l2 = fold(f1), fold(l1), fold(f2), fold(l2)
    fk, lk = token_match(f1, f2), token_match(l1, l2)
    if fk and lk and not (fk != "exact" and lk != "exact"):
        return lk if fk == "exact" else fk if lk == "exact" else None
    sf, sl = token_match(f1, l2), token_match(l1, f2)       # first and last name written the other way round
    if sf and sl and not (sf != "exact" and sl != "exact"):
        return "swapped"
    return None


def name_key(first, last):
    """Same key for 'Ana Lopez', 'Lopez Ana' and 'ana lópez'. Used to count how common a name is."""
    return "|".join(sorted([fold(first) or "", fold(last) or ""]))


name_match_udf = F.udf(name_match_kind, StringType())
name_key_udf = F.udf(name_key, StringType())
fold_udf = F.udf(fold, StringType())
print("name comparison ready")

In [ ]:
# ---- Load clients and prepare matching keys -------------------------------------------
# silver_crm_client holds every VERSION of a client (a change of email, phone or consent is a new row).
# - Names and consent come from the LATEST version of each client.
# - Emails and phones come from ALL versions: if a client used an email last year, it still counts as evidence.
started = now_utc()
versions = spark.table("silver_crm_client")
n_versions = versions.count()
latest_w = Window.partitionBy("client_id").orderBy(F.col("updated_at").desc(), F.col("file_date").desc(), F.col("source_file").desc())
cl = (versions.withColumn("rk", F.row_number().over(latest_w)).filter("rk = 1")
      .select("client_id", "region", "first_name", "last_name", "created_at",
              "consent_share_cross_region", "consent_share_missing")
      .withColumn("f_norm", fold_udf("first_name")).withColumn("l_norm", fold_udf("last_name"))
      .withColumn("name_block", F.soundex(F.col("l_norm")))
      .withColumn("name_block_swap", F.soundex(F.col("f_norm")))
      .withColumn("name_key", name_key_udf("first_name", "last_name")))
# How many clients carry each name? A pair that matches only on a very common name ("Yuto Ito") is weak evidence.
cl = cl.join(cl.groupBy("name_key").agg(F.count("*").alias("name_holders")), "name_key")
n_clients = cl.count()
add_dq("silver_crm_client", "client_has_several_versions", "info", n_clients,
       versions.groupBy("client_id").count().filter("count > 1").count())

def identifiers(col):
    # one row per (client, value) over all versions; values that are null or empty are dropped
    return (versions.filter(F.col(col).isNotNull() & (F.trim(F.col(col)) != ""))
            .select("client_id", "region", F.col(col).alias("value")).distinct())

ids_email, ids_phone = identifiers("email_hash"), identifiers("phone_e164")

# Shared identifiers: used by more than 2 clients, or by 2 clients in the same region.
def shared_values(ids):
    g = ids.groupBy("value").agg(F.countDistinct("client_id").alias("n"), F.countDistinct("region").alias("regions"))
    return g.filter((F.col("n") > 2) | (F.col("n") > F.col("regions"))).select("value")

shared_email, shared_phone = shared_values(ids_email), shared_values(ids_phone)
print("clients:", n_clients, "| versions:", n_versions, "| shared emails:", shared_email.count(), "| shared phones:", shared_phone.count())

In [ ]:
# ---- Candidate pairs ------------------------------------------------------------------
# A pair is two records from different regions. To avoid comparing everyone with everyone,
# records are only compared when they share a "blocking key": an email, a phone, or a sound-alike last name.
def id_pairs(ids, shared):
    x, y = ids.alias("x"), ids.alias("y")
    p = (x.join(y, (F.col("x.value") == F.col("y.value")) & (F.col("x.region") != F.col("y.region")) & (F.col("x.client_id") < F.col("y.client_id")))
         .select(F.col("x.client_id").alias("client_id_a"), F.col("y.client_id").alias("client_id_b"), F.col("x.value").alias("value")))
    return (p.join(shared.withColumn("is_shared", F.lit(True)), "value", "left")
            .groupBy("client_id_a", "client_id_b").agg(F.max(F.coalesce("is_shared", F.lit(False))).alias("shared")))

email_pairs = id_pairs(ids_email, shared_email).withColumnRenamed("shared", "email_shared")
phone_pairs = id_pairs(ids_phone, shared_phone).withColumnRenamed("shared", "phone_shared")

a, b = cl.alias("a"), cl.alias("b")
name_pairs = (a.join(b, ((F.col("a.name_block") == F.col("b.name_block")) |
                         (F.col("a.name_block") == F.col("b.name_block_swap")) |
                         (F.col("a.name_block_swap") == F.col("b.name_block"))) &
                        (F.col("a.region") != F.col("b.region")) & (F.col("a.client_id") < F.col("b.client_id")))
              .select(F.col("a.client_id").alias("client_id_a"), F.col("b.client_id").alias("client_id_b")))

keys = reduce(lambda x, y: x.unionByName(y), [email_pairs.select("client_id_a", "client_id_b"),
                                                phone_pairs.select("client_id_a", "client_id_b"), name_pairs]).distinct()
pick = lambda alias: [F.col(f"{alias}.client_id").alias(f"client_id_{alias}"), F.col(f"{alias}.region").alias(f"region_{alias}"),
                      F.col(f"{alias}.first_name").alias(f"first_{alias}"), F.col(f"{alias}.last_name").alias(f"last_{alias}"),
                      F.col(f"{alias}.name_holders").alias(f"holders_{alias}")]
pairs = (keys.join(a, keys.client_id_a == F.col("a.client_id")).join(b, keys.client_id_b == F.col("b.client_id"))
         .select(*pick("a"), *pick("b"))
         .join(email_pairs, ["client_id_a", "client_id_b"], "left").join(phone_pairs, ["client_id_a", "client_id_b"], "left")
         .withColumn("email_match", F.col("email_shared").isNotNull())
         .withColumn("phone_match", F.col("phone_shared").isNotNull())
         .withColumn("shared_identifier", F.coalesce(F.col("email_shared"), F.lit(False)) | F.coalesce(F.col("phone_shared"), F.lit(False)))
         .withColumn("name_kind", name_match_udf("first_a", "last_a", "first_b", "last_b"))
         .withColumn("name_ok", F.col("name_kind").isNotNull())
         .withColumn("name_popularity", F.greatest("holders_a", "holders_b")))

rule = (F.when(F.col("email_match") & F.col("phone_match"), "email_and_phone")
         .when(F.col("email_match") & F.col("name_ok"), "email_and_name")
         .when(F.col("phone_match") & F.col("name_ok"), "phone_and_name")
         .when(F.col("email_match"), "email_only")
         .when(F.col("phone_match"), "phone_only")
         .when(F.col("name_ok") & (F.col("name_popularity") <= RARE_NAME_MAX), "name_only_rare_name")
         .when(F.col("name_ok"), "name_only_common_name"))
conf = F.lit(None).cast("double")
for r, c in RULE_CONFIDENCE.items():
    conf = F.when(F.col("match_rule") == r, F.lit(c)).otherwise(conf)

pairs = (pairs.withColumn("match_rule", rule).filter(F.col("match_rule").isNotNull())
         .withColumn("confidence", conf)
         .withColumn("confidence", F.when(F.col("shared_identifier") & (F.col("confidence") > SHARED_IDENTIFIER_CONFIDENCE),
                                          F.lit(SHARED_IDENTIFIER_CONFIDENCE)).otherwise(F.col("confidence")))
         .withColumn("decision", F.when(F.col("confidence") >= AUTO_LINK_MIN, "linked").otherwise("review"))
         .select("client_id_a", "client_id_b", "region_a", "region_b", "match_rule", "confidence", "decision",
                 "email_match", "phone_match", "name_kind", "name_popularity", "shared_identifier",
                 "first_a", "last_a", "first_b", "last_b"))
pairs = pairs.cache()
print("candidate pairs scored:", pairs.count())
pairs.groupBy("match_rule", "decision").count().orderBy("match_rule").show(truncate=False)

In [ ]:
# ---- Group linked records into people -------------------------------------------------
# Only a few percent of clients are in a link, so the grouping is done on the driver (union-find).
# For millions of links this step would move to a graph library or an iterative Spark job.
started_group = now_utc()
links = [(r["client_id_a"], r["client_id_b"]) for r in pairs.filter("decision = 'linked'").select("client_id_a", "client_id_b").collect()]
region_of = {r["client_id"]: r["region"] for r in cl.select("client_id", "region").collect()}

parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
for x, y in links:
    rx, ry = find(x), find(y)
    if rx != ry:
        parent[max(rx, ry)] = min(rx, ry)          # the smallest client_id is the group's anchor

groups = {}
for cid in list(parent):
    groups.setdefault(find(cid), []).append(cid)

# A person has at most one record per region. A group that breaks this is not trusted: it is split up.
conflict_groups = {a: m for a, m in groups.items() if len({region_of[c] for c in m}) < len(m)}
split_clients = {c for m in conflict_groups.values() for c in m}
good_groups = {a: m for a, m in groups.items() if a not in conflict_groups}
anchor_of = {c: a for a, m in good_groups.items() for c in m}
print(f"people with 2+ records: {len(good_groups)} | groups split because of a same-region clash: {len(conflict_groups)}")
add_dq("silver_identity_pairs", "linked_group_has_two_records_in_one_region", "reject", len(groups), len(conflict_groups))

pairs = pairs.withColumn("decision", F.when(F.col("client_id_a").isin(list(split_clients)) & (F.col("decision") == "linked"),
                                            "rejected_same_region").otherwise(F.col("decision")))

In [ ]:
# ---- Write the pair table and the client -> person map --------------------------------
started = now_utc()
anchor_df = spark.createDataFrame([(c, a) for c, a in anchor_of.items()], ["client_id", "anchor_id"])

# For each record: the best link it takes part in, and the weakest link its group relies on.
linked = pairs.filter("decision = 'linked'")
ends = (linked.select(F.col("client_id_a").alias("client_id"), "match_rule", "confidence")
        .unionByName(linked.select(F.col("client_id_b").alias("client_id"), "match_rule", "confidence")))
best = (ends.withColumn("rk", F.row_number().over(Window.partitionBy("client_id").orderBy(F.col("confidence").desc(), F.col("match_rule"))))
        .filter("rk = 1").select("client_id", F.col("match_rule").alias("link_rule"), F.col("confidence").alias("best_link_confidence")))

ident = (cl.join(anchor_df, "client_id", "left").join(best, "client_id", "left")
         .withColumn("anchor_id", F.coalesce("anchor_id", "client_id"))
         .withColumn("person_key", F.concat(F.lit("PK-"), F.substring(F.sha2(F.col("anchor_id"), 256), 1, 12))))
w = Window.partitionBy("person_key")
ident = (ident.withColumn("cluster_size", F.count("*").over(w))
         .withColumn("regions_in_cluster", F.size(F.collect_set("region").over(w)))
         .withColumn("weakest_link_confidence", F.min("best_link_confidence").over(w))
         .withColumn("link_rule", F.coalesce("link_rule", F.lit("no_match")))
         .withColumn("is_anchor", F.col("client_id") == F.col("anchor_id")))
ident_out = ident.select("client_id", "region", "person_key", "cluster_size", "regions_in_cluster", "link_rule",
                         F.col("weakest_link_confidence").alias("link_confidence"), "is_anchor",
                         F.lit(SILVER_BATCH_ID).alias("silver_batch_id"))
n_ident = write_table(ident_out, "silver_client_identity")
n_pairs = write_table(pairs.withColumn("silver_batch_id", F.lit(SILVER_BATCH_ID)), "silver_identity_pairs")
log_run("silver_identity_pairs", "success", n_pairs, started, f"pairs={n_pairs}")
log_run("silver_client_identity", "success", n_ident, started, f"rows={n_ident}")
print("silver_client_identity:", n_ident, "rows | silver_identity_pairs:", n_pairs, "rows")

In [ ]:
# ---- One row per person, with the consent decision ------------------------------------
# Rule: cross-region sharing is allowed only if EVERY record of the person says yes.
# A missing answer was already turned into "no" in nb_20, so it also blocks sharing.
started = now_utc()
person = (spark.table("silver_client_identity").join(cl.select("client_id", "created_at", "consent_share_cross_region", "consent_share_missing"), "client_id")
          .groupBy("person_key")
          .agg(F.count("*").alias("record_count"),
               F.sort_array(F.collect_set("region")).alias("regions"),
               F.min("created_at").alias("first_seen_at"),
               (F.countDistinct("consent_share_cross_region") > 1).alias("consent_conflict"),
               F.min(F.col("consent_share_cross_region").cast("int")).cast("boolean").alias("consent_share_effective"),
               F.max("consent_share_missing").alias("consent_share_any_missing"))
          .withColumn("is_multi_region", F.col("record_count") > 1)
          .withColumn("silver_batch_id", F.lit(SILVER_BATCH_ID)))
n_person = write_table(person, "silver_person")
log_run("silver_person", "success", n_person, started, f"rows={n_person}")
multi = spark.table("silver_person").filter("is_multi_region")
add_dq("silver_person", "consent_conflict_between_regions", "info", multi.count(), multi.filter("consent_conflict").count())
add_dq("silver_identity_pairs", "pair_left_for_review", "info", n_pairs, spark.table("silver_identity_pairs").filter("decision = 'review'").count())
print("silver_person:", n_person, "people")

In [ ]:
# ---- Reconciliation, then write control tables ---------------------------------------
# 1) every client record appears exactly once in the identity map
# 2) the people table accounts for every record
rec_clients = n_clients
rec_ident = spark.table("silver_client_identity").count()
rec_ident_distinct = spark.table("silver_client_identity").select("client_id").distinct().count()
rec_people_records = spark.table("silver_person").agg(F.sum("record_count")).collect()[0][0]
for name, ok in [("clients vs identity map", rec_clients == rec_ident == rec_ident_distinct),
                 ("identity map vs people", rec_ident == rec_people_records)]:
    if not ok:
        gaps.append(name)

DQ_SCHEMA = StructType([
    StructField("batch_id", StringType()), StructField("layer", StringType()),
    StructField("table_name", StringType()), StructField("rule", StringType()),
    StructField("severity", StringType()), StructField("rows_checked", LongType()),
    StructField("rows_failed", LongType()), StructField("pct_failed", DoubleType()),
    StructField("checked_at", TimestampType())])
RUN_SCHEMA = StructType([
    StructField("batch_id", StringType()), StructField("layer", StringType()),
    StructField("object_name", StringType()), StructField("status", StringType()),
    StructField("files_loaded", IntegerType()), StructField("rows_loaded", LongType()),
    StructField("started_at", TimestampType()), StructField("finished_at", TimestampType()),
    StructField("failure_class", StringType()), StructField("message", StringType())])
spark.createDataFrame(dq_rows, schema=DQ_SCHEMA).write.format("delta").mode("append") \
     .option("mergeSchema", "true").saveAsTable("ctl_dq_results")
spark.createDataFrame(run_rows, schema=RUN_SCHEMA).write.format("delta").mode("append") \
     .option("mergeSchema", "true").saveAsTable("ctl_run_log")

if gaps:
    raise RuntimeError("Identity reconciliation failed for: " + ", ".join(gaps))
print("silver part 3 finished, batch", SILVER_BATCH_ID)

## Check the result
Read-only cells. Run them after the load.

In [ ]:
# Reconciliation: every client record is in the map once, and the people table covers all records
spark.createDataFrame([("clients (latest version of each)", rec_clients), ("rows in silver_client_identity", rec_ident),
                       ("distinct client_id in identity map", rec_ident_distinct), ("records covered by silver_person", int(rec_people_records)),
                       ("people", n_person)], ["check", "value"]).show(truncate=False)

In [ ]:
# How many pairs each rule produced, and what was decided
(spark.table("silver_identity_pairs").groupBy("match_rule", "confidence", "decision").count()
      .orderBy(F.col("confidence").desc(), "decision").show(truncate=False))

In [ ]:
# People by number of regions
(spark.table("silver_person").groupBy("record_count", "is_multi_region").count().orderBy("record_count").show())

In [ ]:
# Eyeball linked pairs: names, how the name compared, and which rule linked them
(spark.table("silver_identity_pairs").filter("decision = 'linked'")
      .select("region_a", "first_a", "last_a", "region_b", "first_b", "last_b", "name_kind", "match_rule", "confidence")
      .orderBy("name_kind").show(15, truncate=False))

In [ ]:
# Pairs left for review (not merged)
(spark.table("silver_identity_pairs").filter("decision = 'review'")
      .select("region_a", "first_a", "last_a", "region_b", "first_b", "last_b", "name_kind", "match_rule", "confidence")
      .orderBy(F.col("confidence").desc()).show(10, truncate=False))

In [ ]:
# Consent conflicts: people whose regions disagree about cross-region sharing
(spark.table("silver_person").filter("is_multi_region")
      .groupBy("consent_conflict", "consent_share_effective").count().orderBy("consent_conflict").show())